In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the window length and switch the shape. On the left, the gold curve
is the window and the red curve is what the transform actually sees. On
the right is the resulting amplitude spectrum in decibels, with the two
true frequencies marked in gray. A shorter window widens every peak, and
the Hann window trades a wider peak for much smaller side lobes.

In [ ]:
# hide
# autorun
T0, SHAPE0 = 4.0, "Rectangular"     # starting parameters

SR_A = 200.0                        # analysis rate for the running example
T_TOT = 8.0
TA = np.arange(int(T_TOT * SR_A)) / SR_A
X = np.sin(2 * np.pi * TA) + np.sin(2 * np.pi * 2 * TA)
NFFT = 8192
FREQS = np.fft.rfftfreq(NFFT, 1 / SR_A)
MASK = FREQS <= 5.0

def window(T, shape):
    w = np.zeros_like(TA)
    inside = TA < T
    if shape == "Hann":
        w[inside] = 0.5 * (1 - np.cos(2 * np.pi * TA[inside] / T))
    else:
        w[inside] = 1.0
    return w

def spectrum(T, shape):
    mag = np.abs(np.fft.rfft(X * window(T, shape), NFFT))[MASK]
    return 20 * np.log10(np.maximum(mag / mag.max(), 1e-4))

def figure():
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.13)
    w0 = window(T0, SHAPE0)
    fig.add_scatter(x=TA, y=X, mode="lines",
                    line=dict(color=STEEL, width=1.2), row=1, col=1)
    fig.add_scatter(x=TA, y=2.2 * w0, mode="lines",
                    line=dict(color=GOLD, width=1.8, dash="dash"),
                    row=1, col=1)
    fig.add_scatter(x=TA, y=X * w0, mode="lines",
                    line=dict(color=RED, width=1.6), row=1, col=1)
    fig.add_scatter(x=[1, 1, None, 2, 2, None], y=[-82, 4, None, -82, 4, None],
                    mode="lines", line=dict(color=STEEL, width=1.2, dash="dot"),
                    row=1, col=2)
    fig.add_scatter(x=FREQS[MASK], y=spectrum(T0, SHAPE0), mode="lines",
                    line=dict(color=BLUE, width=1.6), row=1, col=2)
    fig.update_xaxes(range=[0, T_TOT], title_text="Time (s)",
                     fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[-2.6, 2.6], title_text="Amplitude",
                     fixedrange=True, row=1, col=1)
    fig.update_xaxes(range=[0, 5], title_text="Frequency (Hz)",
                     fixedrange=True, row=1, col=2)
    fig.update_yaxes(range=[-82, 4], title_text="Magnitude (dB)",
                     fixedrange=True, row=1, col=2)
    return fig

def controls(fig):
    length = widgets.FloatSlider(description="Window length T (s)", min=0.5,
                                 max=8.0, value=T0, step=0.1)
    shape = widgets.Dropdown(description="Window shape",
                             options=["Rectangular", "Hann"], value=SHAPE0)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(T, shape, X=X, window=window, spectrum=spectrum,
               readout=readout):
        w = window(T, shape)
        with fig.batch_update():
            fig.data[1].y = 2.2 * w
            fig.data[2].y = X * w
            fig.data[4].y = spectrum(T, shape)
        readout.value = (f"<span style='font-size:0.9em'>T = {T:.1f} s "
                         f"&nbsp;·&nbsp; bin spacing 1/T = {1 / T:.2f} Hz"
                         f"</span>")

    widgets.interactive_output(update, {"T": length, "shape": shape})
    return widgets.VBox([length, shape, readout])

icm_plotly.show(figure, controls)